# Lecture 12 — Hardware Acceleration on the CPU

**CMU 10-414/714, Deep Learning Systems (lecture by Tianqi Chen)**
Companion notebook · Sangram Lembe

Fast matrix-multiply libraries mostly do the *same* $n^3$ multiplications as the textbook triple
loop. They are fast because they fetch each number from slow memory far fewer times. This notebook
measures that claim instead of repeating it:

1. **vectorisation** — one instruction, many numbers
2. **memory layout and strides** — transpose, slice and broadcast without copying anything
3. **access order** — the same data read in a different order runs at a different speed
4. **tiling** — a cache simulator shows identical arithmetic with 8× fewer cache misses

In [1]:
import numpy as np, time, os
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import as_strided

def timeit(fn, repeats=5):
    fn()
    best = float("inf")
    for _ in range(repeats):
        t = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t)
    return best

print("CPU cores available:", os.cpu_count())

CPU cores available: 1


## 1. Vectorisation

A plain loop adds one pair of numbers per step. NumPy hands the whole array to compiled code that
uses **vector registers** — several numbers per instruction. With 4 floats per register, 256
additions take 64 steps.

In [2]:
n = 1_000_000
a, b = np.random.rand(n), np.random.rand(n)

def scalar_add():
    c = np.empty(n)
    for i in range(n):
        c[i] = a[i] + b[i]
    return c

t_scalar = timeit(scalar_add, repeats=1)
t_vector = timeit(lambda: a + b)
print(f"scalar Python loop : {t_scalar*1e3:9.1f} ms")
print(f"vectorised a + b   : {t_vector*1e3:9.2f} ms")
print(f"speed-up           : {t_scalar/t_vector:9.0f}x")

scalar Python loop :     198.4 ms
vectorised a + b   :      0.86 ms
speed-up           :       232x


Most of that gap is Python interpreter overhead rather than vector instructions alone — but both
come from the same place: doing one thing at a time.

### Alignment

Vector loads usually require the start address to be a multiple of the vector width (16 bytes for
four floats). NumPy allocates with strong alignment for exactly this reason. A slice that starts one
element in breaks it.

In [3]:
x = np.zeros(1024, dtype=np.float32)
print("fresh array address mod 16 :", x.ctypes.data % 16, "  <- aligned")
print("x[1:] address mod 16       :", x[1:].ctypes.data % 16, "  <- 4 bytes off: misaligned")

fresh array address mod 16 : 0   <- aligned
x[1:] address mod 16       : 4   <- 4 bytes off: misaligned


## 2. Memory layout

Memory is one long line of addresses. A $2\times3$ matrix has to be laid out in some order.

In [4]:
A = np.arange(6).reshape(2, 3)
print(A, "\n")
print("row-major (C / NumPy)      :", A.ravel(order="C"))
print("column-major (Fortran/BLAS):", A.ravel(order="F"))
print("\nstrides of A, in elements  :", tuple(s // A.itemsize for s in A.strides))

[[0 1 2]
 [3 4 5]] 

row-major (C / NumPy)      : [0 1 2 3 4 5]
column-major (Fortran/BLAS): [0 3 1 4 2 5]

strides of A, in elements  : (3, 1)


**Strides** say how far to jump in memory when an index increases by one. The address of an element
is just a dot product:

$$\text{address}(i_0, i_1, \ldots) = \text{offset} + \sum_k i_k \cdot \text{stride}_k$$

Implementing that directly and checking it against NumPy on arrays of every kind:

In [5]:
def element(buf, offset, strides, idx):
    """Read one element from a flat buffer using offset + sum(i_k * stride_k)."""
    return buf[offset + sum(i * s for i, s in zip(idx, strides))]

base = np.arange(24.0)
flat = base                                                    # the raw memory
views = {
    "row-major 4x6"   : (0, (6, 1),  (4, 6)),
    "column-major 4x6": (0, (1, 4),  (4, 6)),
    "transpose of 4x6": (0, (1, 6),  (6, 4)),
    "slice [1:, 2:]"  : (8, (6, 1),  (3, 4)),
    "every other col" : (0, (6, 2),  (4, 3)),
}
for name, (off, st, shape) in views.items():
    ref = as_strided(base[off:], shape=shape, strides=tuple(s * 8 for s in st))
    ok = all(element(flat, off, st, idx) == ref[idx] for idx in np.ndindex(*shape))
    print(f"{name:<18} strides {str(st):<8} matches NumPy: {ok}")

row-major 4x6      strides (6, 1)   matches NumPy: True
column-major 4x6   strides (1, 4)   matches NumPy: True
transpose of 4x6   strides (1, 6)   matches NumPy: True
slice [1:, 2:]     strides (6, 1)   matches NumPy: True
every other col    strides (6, 2)   matches NumPy: True


### Three operations that copy nothing

Because an array is just (memory, offset, shape, strides), several operations only change those
numbers:

| operation | what changes |
|---|---|
| transpose | swap the strides |
| slice | move the offset |
| broadcast | set a stride to **0** |

In [6]:
A = np.arange(6.0).reshape(2, 3)
el = lambda arr: tuple(s // arr.itemsize for s in arr.strides)

T = A.T
S = A[:, 1:]
B = np.broadcast_to(np.array([7.0, 8.0, 9.0]), (4, 3))

for name, arr, parent in [("transpose", T, A), ("slice A[:,1:]", S, A)]:
    print(f"{name:<15} strides {el(arr)}   shares memory with A: {np.shares_memory(arr, parent)}")
print(f"{'broadcast':<15} strides {el(B)}   <- stride 0: every row reads the same 3 numbers")
print(B)

transpose       strides (1, 3)   shares memory with A: True
slice A[:,1:]   strides (3, 1)   shares memory with A: True
broadcast       strides (0, 1)   <- stride 0: every row reads the same 3 numbers
[[7. 8. 9.]
 [7. 8. 9.]
 [7. 8. 9.]
 [7. 8. 9.]]


### The catch: non-contiguous arrays

The slice above is no longer one neat block. Routines like matrix multiply and vectorised loads want
compact memory, so libraries offer `contiguous()` / `compact()` — copy only when needed.

In [7]:
def compact(arr):
    """Return arr unchanged if already contiguous, otherwise a compact copy."""
    return arr if arr.flags["C_CONTIGUOUS"] else np.ascontiguousarray(arr)

for name, arr in [("A", A), ("A.T", A.T), ("A[:, 1:]", A[:, 1:])]:
    c = compact(arr)
    print(f"{name:<9} contiguous: {str(arr.flags['C_CONTIGUOUS']):<5}  compact() copied: {c is not arr}")

A         contiguous: True   compact() copied: False
A.T       contiguous: False  compact() copied: True
A[:, 1:]  contiguous: False  compact() copied: True


## 3. Access order changes speed, with the same data

NumPy arrays are row-major, so a row is contiguous and a column jumps a whole row length between
elements. Summing all rows and summing all columns does the same arithmetic on the same numbers:

In [8]:
M = np.random.rand(4000, 4000)

def by_rows():
    return sum(M[i, :].sum() for i in range(M.shape[0]))

def by_cols():
    return sum(M[:, j].sum() for j in range(M.shape[1]))

t_r, t_c = timeit(by_rows), timeit(by_cols)
print(f"row-by-row (contiguous) : {t_r*1e3:6.1f} ms")
print(f"column-by-column (strided): {t_c*1e3:6.1f} ms   -> {t_c/t_r:.1f}x slower")

row-by-row (contiguous) :   18.5 ms
column-by-column (strided):   35.7 ms   -> 1.9x slower


Reading contiguous memory uses every byte of each cache line fetched; striding down a column uses one
number per line and throws the rest away.

## 4. Parallelisation

Splitting independent loop iterations across cores is the third general technique.

In [9]:
print("cores on this machine:", os.cpu_count())

cores on this machine: 1


This environment has a single core, so a parallel speed-up cannot be demonstrated honestly here — any
timing would only show threading overhead. On a four-core machine, an OpenMP `parallel for` splits a
loop into four chunks, one per core.

## 5. Matrix multiplication: count the loads

$C_{ij} = \sum_k A_{ik} B_{jk}$. The textbook loop is $O(n^3)$, and so are the fast libraries. The
difference is memory: fetching from main memory is roughly 200× slower than from L1 cache. So count
**loads** from slow memory, exactly as the lecture does.

In [10]:
def naive_loads(n):
    loadsA = loadsB = 0
    for i in range(n):
        for j in range(n):
            for k in range(n):
                loadsA += 1; loadsB += 1
    return loadsA + loadsB, 3                           # loads, registers used

def register_tiled_loads(n, v1, v2, v3=1):
    loadsA = loadsB = 0
    for i in range(n // v1):
        for j in range(n // v2):
            for k in range(n // v3):
                loadsA += v1 * v3                       # a strip of A into registers
                loadsB += v2 * v3                       # a strip of B into registers
    return loadsA + loadsB, v1*v3 + v2*v3 + v1*v2

n = 64
print(f"{'scheme':<22}{'loads (counted)':>16}{'formula':>12}{'registers':>11}")
print("-" * 61)
L, R = naive_loads(n)
print(f"{'naive':<22}{L:>16,}{2*n**3:>12,}{R:>11}")
for v in (2, 4, 8):
    L, R = register_tiled_loads(n, v, v)
    print(f"{'register tile %dx%d' % (v, v):<22}{L:>16,}{n**3//v + n**3//v:>12,}{R:>11}")

scheme                 loads (counted)     formula  registers
-------------------------------------------------------------
naive                          524,288     524,288          3
register tile 2x2              262,144     262,144          8
register tile 4x4              131,072     131,072         24
register tile 8x8               65,536      65,536         80


The counted loads match $n^3/v_1 + n^3/v_2$ exactly. A $4\times4$ tile gives four times fewer loads
using 24 registers; $8\times8$ gives eight times fewer but needs 80 registers — more than a typical
CPU has. The tile must fit the hardware.

And the arithmetic really is unchanged — the tiled loop computes the same product:

In [11]:
def matmul_register_tiled(A, B, v1=4, v2=4):
    """C = A @ B.T, computing a v1 x v2 tile of C at a time."""
    n = A.shape[0]
    C = np.zeros((n, n))
    for i in range(0, n, v1):
        for j in range(0, n, v2):
            c = np.zeros((v1, v2))                  # the tile lives in 'registers'
            for k in range(n):
                a = A[i:i+v1, k]                    # v1 loads, each reused v2 times
                b = B[j:j+v2, k]                    # v2 loads, each reused v1 times
                c += np.outer(a, b)
            C[i:i+v1, j:j+v2] = c
    return C

A, B = np.random.rand(32, 32), np.random.rand(32, 32)
print("matches A @ B.T:", np.allclose(matmul_register_tiled(A, B), A @ B.T))

matches A @ B.T: True


## 6. Seeing the cache: a simulator

Load counts assume a perfect two-level machine. To see a real cache effect, simulate one: a small
fully-associative LRU cache holding 64 lines of 8 doubles (4 KB), fed every address a matrix
multiply touches. Same accesses in every run — only the **order** changes.

In [12]:
from collections import OrderedDict
class LRUCache:
    """Fully associative LRU cache. Counts misses on 64-byte lines (8 doubles)."""
    def __init__(self, n_lines, line_elems=8):
        self.n, self.le, self.lines, self.misses, self.accesses = n_lines, line_elems, OrderedDict(), 0, 0
    def access(self, addr):
        self.accesses += 1
        line = addr // self.le
        if line in self.lines:
            self.lines.move_to_end(line)
        else:
            self.misses += 1
            self.lines[line] = True
            if len(self.lines) > self.n:
                self.lines.popitem(last=False)

def naive(n, cache):
    for i in range(n):
        for j in range(n):
            for k in range(n):
                cache.access(i*n + k)          # A[i,k]
                cache.access(n*n + j*n + k)    # B[j,k]

def tiled(n, cache, b):
    for i0 in range(0, n, b):
        for j0 in range(0, n, b):
            for k0 in range(0, n, b):
                for i in range(i0, i0+b):
                    for j in range(j0, j0+b):
                        for k in range(k0, k0+b):
                            cache.access(i*n + k)
                            cache.access(n*n + j*n + k)


n = 64
print(f"{'loop order':<16}{'accesses':>12}{'cache misses':>15}")
print("-" * 43)
c = LRUCache(64); naive(n, c)
print(f"{'naive':<16}{c.accesses:>12,}{c.misses:>15,}")
for b in (4, 8, 16, 32):
    c = LRUCache(64); tiled(n, c, b)
    print(f"{'tiled, b = %d' % b:<16}{c.accesses:>12,}{c.misses:>15,}")

loop order          accesses   cache misses
-------------------------------------------
naive                524,288         33,280


tiled, b = 4         524,288         14,464

tiled, b = 8         524,288          8,192
tiled, b = 16        524,288          4,096


tiled, b = 32        524,288         33,792


Identical arithmetic, identical number of memory accesses — and tiling with $b = 16$ cuts cache misses
from 33,280 to 4,096, eight times fewer. Push the tile too large ($b = 32$) and the working set no
longer fits in the 64-line cache, so misses rise again. That is the lecture's constraint in action:
make tiles as large as possible, **but no larger than the cache**.

## 7. Why libraries win

Everything above is what an optimised BLAS does in compiled code, with vector instructions, several
cache levels and every core. Compare it against the plain triple loop:

In [13]:
def triple_loop(A, B):
    n = A.shape[0]
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            s = 0.0
            for k in range(n):
                s += A[i, k] * B[j, k]
            C[i, j] = s
    return C

n = 96
A, B = np.random.rand(n, n), np.random.rand(n, n)
t_loop = timeit(lambda: triple_loop(A, B), repeats=1)
t_blas = timeit(lambda: A @ B.T)
print(f"triple loop : {t_loop*1e3:10.1f} ms")
print(f"BLAS (A@B.T): {t_blas*1e3:10.3f} ms   -> {t_loop/t_blas:,.0f}x faster, same O(n^3) maths")

triple loop :      229.9 ms
BLAS (A@B.T):      0.038 ms   -> 6,118x faster, same O(n^3) maths


Part of that gap is Python itself, but the rest is exactly the techniques in this lecture:
vectorisation, tiling for registers and cache, and parallelism.

## 8. The reuse rule

In $C_{ij} = \sum_k A_{ik}B_{jk}$, $A_{ik}$ does not use $j$. So while $j$ varies, the same $A$ value is
needed again — tile along $j$ and it can stay in fast memory. $B_{jk}$ does not use $i$, so tile along
$i$. **A missing index means reuse.**

In [14]:
n = 8
uses_A = {}
for i in range(n):
    for j in range(n):
        for k in range(n):
            uses_A[(i, k)] = uses_A.get((i, k), 0) + 1
print("each A[i,k] is needed", set(uses_A.values()), "times - once for every j")

each A[i,k] is needed {8} times - once for every j


## Summary

- Vectorisation processes several numbers per instruction and needs aligned memory.
- An array is memory + offset + shape + strides. Transpose, slice and broadcast only change those
  numbers — verified to share memory.
- Reading the same data column-wise instead of row-wise was roughly 1.7–1.8× slower here, purely from access order.
- Register tiling cuts loads from $2n^3$ to $n^3/v_1 + n^3/v_2$ — the counted numbers match exactly.
- In a simulated cache, tiling cut misses 8× with identical arithmetic, until the tile outgrew the cache.